In [ ]:
### 선호/비선호 데이터 셋을 통한 DPO 지시 학습
    - PreferenceDataset class
    - custom_collate_fn()
    - loss : compute_logprobs(), calc_dpo_loss_batch(), calc_dpo_loss()
    - train : train_dpo_simple()
    - DPO 를 통한 Policy 모델을 Reference 모델로 선호쪽으로 학습 튜닝 / 학습 / 평가

In [ ]:
### PreferenceDataset class
    - DPO 용 프롬프트, 선택된 답변, 거부된 답변 쌍 필요

class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []

        ## 1. 데이터를 미리 토큰화하여 저장
        for entry in data:
            prompt = format_input( entry )
            rejected_response = entry['rejected']   # 비선호 답변
            chosen_response = entry['chosen']       # 선호 답변

            ## 1.1 프롬프트만 따로 인코딩
            prompt_tokens = tokenizer.encode( prompt )  # 나중에 마스킹 용도로 사용

            ## 1.2 프롬프트 + 답변 형태로 전체 문장을 인코딩
            chosen_full_tokens = tokenizer.encode( f"{prompt}\n\n### Response:\n{chosen_response}" )
            rejected_full_tokens = tokenizer.encode( f"{prompt}\n\n### Response:\n{rejected_response}" )

            ## 1.3 최종 토큰 목록에 추가 (json 형식)
            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __len__():
        return len(self.data)

    def __getitem__(self, index):
        return self.encoded_texts[index]

In [ ]:
### DPO 용 custom_collate_fn

def custom_collate_fn( batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu" ):
    """
        배치 내의 데이터 길이를 맞추고(padding), 마스크(mask)를 생성하는 함수
    """

    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "chosen_mask": [],
        "rejected_mask": []
    }

    ## 1. 배치 내에서 가장 긴 시퀀스의 길이 찾기
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max( len(item[key]) + 1 for item in batch )
            max_length_common = max( max_length_common, current_max )

    ## 2. 패딩 추가 및 마스크 생성
    for item in batch:
        prompt = torch.tensor( item["prompt"] )
        batch_data["prompt"].append( prompt )

        for key in ["chosen", "rejected"]:
            sequence = item[key]
            
            # 패딩 추가
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))

            # 기본 마스크: 데이터가 있는 곳은 1, 패딩은 0
            mask = torch.ones( len(paded) ).bool()
            mask[ len(sequence) : ] = False     # 패딩 부분 마스킹

            # [중요] 프롬프트 부분 마스킹
            #  - DPO 는 '답변'의 확률 차이를 학습하므로, 질문(prompt) 부분은 loss 계산에서 제외
            #  - +2는 "\n\n" 때문
            if mask_prompt_tokens:
                mask[ :prompt.shape[0] + 2 ] = False

            batch_data[key].append( torch.tensor(padded) )
            batch_data[f"{key}_mask"].append( mask )

    ## 3. tensor 변환 및 device 이동
    for key in  ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = tensor.stack( batch_data[key] )

        # 길이 제한 체크
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        
        batch_data[key] = tensor_stack.to(device)

    return batch_data

In [ ]:
### DPO Loss 및 Log Probability 계산 함수

def compute_logprobs( logits, labels, selection_mask=None ):
    """
        모델의 출력(logits)과 정답(labels)을 받아서 해당 정답 토큰의 로그 확률을 계산
    """

    ## 1. AR 모델 특성 상 입력 [A, B, C] 에 대한 예측은 [B, C, D]가 되므로 shift 합니다.
    labels = labels[:, 1:].clone()
    logits = logits[:, :-1, :]      # logits 은 마지막 꺼만 가져옴

    log_probs = F.log_softmax( logits, dim=-1 )

    ## 2. 실제 정답 레이블에 해당하는 확률값만 추츨 (gather 사용)
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        # 마스크도 shift 해서 적용 (패딩이나 프롬프트 영역 무시)
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask

        # 유효한 토큰들의 로그 확률 평균 계산
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)


def compute_dpo_loss( model_chosen_logprobs, model_rejected_logprobs,
                      reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1 ):
    """
        DPO 손실 함수 계산:
            Policy 모델이 Reference 모델보다 'chosen' 답변을 더 선호하고, 'rejected' 답변을 덜 선호하도록 유도
            beta : Reference 모델에서 얼마나 벗어날지 제어하는 하이퍼 파라미터 (보통 0.1 ~ 0.5)
    """

    ## 1. Policy 모델의 (chosen - rejected) 로그 확률 차이 계산
    model_logratios = model_chosen_logprobs - model_rejected_logprobs

    ## 2. Reference 모델의 (chosen - rejected) 로그 확률 차이 계산
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs

    ## 3. 두 비율의 차이 (policy가 reference보다 얼마나 더 잘 구분했는가) 계산
    logits = model_logratios - reference_logratios

    ## 4. sigmoid 후 음수 로그 (Cross Entropy 와 유사) -> 이 값을 최소화 하면 선호도 차이가 극대화 됨
    losses = -F.logsigmoid( beta * logits )

    ## 5. 학습 추적용 보상(Rewards) 계산 (로깅용)
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()


def compute_dpo_loss_batch( batch, policy_model, reference_model, beta ):
    """
        하나의 배치에 대해서 전체 DPO 과정을 수행해서 loss 계산
    """

    ## 1. 학습 중인 모델(policy)의 로그 확률 계산 (기울기 계산: O)
    policy_chosen_log_probs = compute_logprobs(
        logits = policy_model( batch["chosen"] )
        labels = batch["chosen"]
        selection_mask = batch["chosen_mask"]
    )

    policy_rejected_log_probs = compute_logprobs(
        logits = policy_model( batch["rejected"] )
        labels = batch["rejected"]
        selection_mask = batch["rejected_mask"]
    )

    ## 2. 기준 모델(reference)의 로그 확률 계산 (기울기 계산: X)
    with torch.no_grad():
        ref_chosen_log_probs = compute_logprobs(
            logits = reference_mode( batch["chosen"] )
            labels = batch["chosen"]
            selection_mask = batch["chosen_mask"]
        )

        ref_rejected_log_probs = compute_logprobs(
            logits = reference_mode( batch["rejected"] )
            labels = batch["rejected"]
            selection_mask = batch["rejected_mask"]
        )

    ## 3. 최종 loss 계산
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs = policy_chosen_log_probs,
        model_rejected_logprobs = policy_rejected_log_probs,
        reference_chosen_logprobs = ref_chosen_log_probs,
        reference_rejected_logprobs = ref_rejected_log_probs,
        beta = beta
    )

    return loss, chosen_rewards, rejected_rewards

In [ ]:
### DPO 모델 학습 및 평가 함수

def train_model_dpo_simple( policy_model, reference_model, train_loader, val_loader, optimizer,
                            num_epochs, beta, eval_freq, eval_iter, start_context, tokenizer):
    """
        전체 학습 과정을 관리하는 함수
    """
    tracking = {"train_losses": [], "val_losses": [], "tokens_seen": []}
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        policy_model.train()     # 학습 모드

        for batch in train_loader:
            optimizer.zero_grad()   # 기울기 초기화

            # loss 계산
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch, policy_model=policy_model, reference_model=reference_model, beta=beta
            )

            loss.backward()     # 역전파
            optimizer.step()    # weight 업데이트

            tokens_seen += batch["chosen"].numel()
            global_step += 1

            # 평가 주기마다 성능 기록
            if global_step % eval_freq == 0 :
                res = evaluate_dpo_loss_loader( policy_model, reference_model, train_loader, val_loader, beta, eval_iter )
                tracking["train_losses"].append( res["train_loss"] )
                tracking["val_losses"].append( res["val_losses"] )
                tracking["tokens_seen"].append( res["tokens_seen"] )

                print( f"Ep {epoch+1} (Step {global_step:06d}): "
                       f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f} "
                       f"Train Margin {res['train_chosen_reward] - res['train_rejected_reward']:.3f}")

    # 에포크가 끝날 때 마다 샘플 생성
    generate_and_print_sample( model=policy_model, tokenizer=tokenizer, device=loss.device, start_context=start_context )

return tracking


def evaluate_dpo_loss_loader( policy_model, reference_model, train_loader, val_loader, beta, eval_iter ) :
    """
        학습 중간에 모델 성능 평가를 하는 함수
    """
    
    policy_model.eval()     # 평가 모드
    with torch.no_grad():
        
        # 로더를 순회하며 평균 loss 를 계산하는 내부 함수
        def compute_loader_metric( loader ):
            total_loss, total_chosen, total_rejected = 0., 0., 0.
            num_batches = min( eval_iter, len(loader) )
            if num_batches == 0 : return float('nan'), float('nan'), float('nan')

            for i, batch in enumerate(loader):
                if i >= num_batches: break

                loss, chosen, rejected = compute_dpo_loss_batch( batch, policy_model, reference_model, beta )
                total_loss += loss.item()
                total_chosen += chosen.item()
                total_rejected += rejected.item()

            return total_loss/num_batches, total_chosen/num_batches, total_rejected/num_batches

        train_loss, train_chosen, train_rejected = compute_loader_metric( train_loader )
        val_loss, val_chosen, val_rejected = compute_loader_metric( val_loader )

    policy_model.train()    # 다시 학습 모드로 되돌림

    return {
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen,
        "train_rejected_reward": train_rejected,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen,
        "val_rejected_reward": val_rejected
    }


In [ ]:
### DPO 용 Policy, Reference 모델 로드 / 학습 / 평가

device = torch.device( "cuda" if torch.cuda.is_available() else "cpu" )
tokenizner = tiktoken.get_encoding( "gpt2" )


## 1. 데이터 로드 및 데이터 분할
file_path = "datas/instruction-data-with-preference.json"
data = download_and_load_file(file_path)

train_portion = int( len(data) * 0.85 )
test_portion = int( len(data) * 0.1 )
train_data = data[ : train_portion ]
test_data = data[ train_portion : train_portion + test_portion ]
val_data = data[ train_portion + test_portion : ]


## 2. 데이터 로드 생성
customized_collate_fn = partial( custom_collate_fn, device=device, mask_prompt_tokens=True, allowed_max_length=1024 )

batch_size = 8
train_loader = DataLoader( PreferenceDataset( train_data, tokenizer ), batch_size=batch_size,
                           collate_fn=customized_collate_fn, shuffle=True, drop_last=True )     # shuffle/drop_last
val_loader = DataLoader( PreferenceDataset( val_data, tokenizer ), batch_size=batch_size,
                         collate_fn=customized_collate_fn, shuffle=False, drop_last=False )

## 3. 모델 설정 (SFT 모델 로드)
BASE_CONFIG = {
    "vocab_size": 50257, "context_length": 1024, "drop_rate": 0.0, "qkv_bias": True,
    "emb_dim": 1024, "n_layers": 24, "n_heads": 16 # gpt2-medium config
}

file_name = "gpt2-medium-355M.pth"
model_path = f"./models/gpt2/{file_name}"
download_model(file_name, model_path)

## 3.1 policy 모델 생성 (학습 대상)
policy_model = GPTModel(BASE_CONFIG)
policy_model.laod_state_dict( torch.load(model_path, map_location="cpu", weights_only=True) )
policy_model.to(device)

## 3.2 reference 모델 생성 (고정, 학습 X)
reference_model = GPTModel(BASE_CONFIG)
reference_model.laod_state_dict( torch.load(model_path, map_location="cpu", weights_only=True) )
reference_model.to(device)
reference_model.eval()


## 4. 학습 시작
optimizer = torch.optim.AdamW( policy_model.parameters(), lr=5e-6, weight_decay=0.01 )

start_time = time.time()
num_epochs = 1

tracking = train_model_dpo_simple(
    policy_model = policy_model,
    reference_model = reference_model,
    train_loader = train_loader,
    val_loader = val_loader,
    optimizer = optimizer,
    num_epochs = num_epochs,
    beta = 0.1
    eval_freq = 5,
    eval_iter = 5,
    start_context = format_input( val_data[2] ),
    tokenizer = tokenizer
)
end_time = time.time()

print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.")